# The Mandelbrot Set

## Mini-project step 1: Naive, NumPy, Numba

### Naive implementation

In [1]:
import numpy as np
import time

def mandelbrot_naive(re_min, re_max, im_min, im_max, p_re, p_im, max_iter):
    M = np.zeros((p_im, p_re))
    
    re_step = (re_max - re_min) / (p_re - 1)
    im_step = (im_max - im_min) / (p_im - 1)
    
    for i in range(p_im):
        c_imag = im_min + i * im_step
        
        for j in range(p_re):
            c_real = re_min + j * re_step
            c = complex(c_real, c_imag)
            
            z = 0.0j
            escape_iter = max_iter 
            
            for k in range(max_iter):
                z = z*z + c
                if abs(z) > 2.0:
                    escape_iter = k + 1
                    break
                    
            M[i, j] = escape_iter / max_iter 
            
    return M

Parameters:

In [2]:
# Fixed boundary parameters
re_min, re_max = -2.0, 1.0
im_min, im_max = -1.5, 1.5

# Grid resolution and maximum iterations
p_re, p_im = 500, 500  
max_iter = 100

Performance parameters:

In [3]:
# Execute the naive function and measure execution time
start_time_naive = time.time()
M_naive = mandelbrot_naive(re_min, re_max, im_min, im_max, p_re, p_im, max_iter)
end_time_naive = time.time()

exec_time_naive = end_time_naive - start_time_naive
print(f"Naive execution time: {exec_time_naive:.4f} seconds")

Naive execution time: 0.6902 seconds


### NumPy implementation

In [4]:
def mandelbrot_vectorized(re_min, re_max, im_min, im_max, p_re, p_im, max_iter):
    re = np.linspace(re_min, re_max, p_re)
    im = np.linspace(im_min, im_max, p_im)
    c = re + 1j * im[:, np.newaxis]
    
    z = np.zeros_like(c)
    M = np.zeros(c.shape, dtype=int)
    
    for k in range(max_iter):
        mask = np.abs(z) <= 2.0
        M[mask] = k + 1
        z[mask] = z[mask]**2 + c[mask]
        
    return M / max_iter

In [5]:
# Execute the vectorized function and measure execution time
start_time_vectorized = time.time()
M_vectorized = mandelbrot_vectorized(re_min, re_max, im_min, im_max, p_re, p_im, max_iter)
end_time_vectorized = time.time()
exec_time_vectorized = end_time_vectorized - start_time_vectorized
print(f"Vectorized execution time: {exec_time_vectorized:.4f} seconds")

Vectorized execution time: 0.3005 seconds


### Numba implementation

In [6]:
from numba import jit

@jit(nopython=True)
def mandelbrot_numba(re_min, re_max, im_min, im_max, p_re, p_im, max_iter):
    M = np.zeros((p_im, p_re))
    
    re_step = (re_max - re_min) / (p_re - 1)
    im_step = (im_max - im_min) / (p_im - 1)
    
    for i in range(p_im):
        c_imag = im_min + i * im_step
        
        for j in range(p_re):
            c_real = re_min + j * re_step
            c = complex(c_real, c_imag)
            
            z = 0.0j
            escape_iter = max_iter 
            
            for k in range(max_iter):
                z = z*z + c
                if abs(z) > 2.0:
                    escape_iter = k + 1
                    break
                    
            M[i, j] = escape_iter / max_iter 
            
    return M

In [7]:
# Warm-up run to allow Numba to compile the code
_ = mandelbrot_numba(re_min, re_max, im_min, im_max, 10, 10, 10)

# Measure execution time
start_time_nb = time.time()
M_numba = mandelbrot_numba(re_min, re_max, im_min, im_max, p_re, p_im, max_iter)
end_time_nb = time.time()

exec_time_nb = end_time_nb - start_time_nb
print(f"Numba execution time: {exec_time_nb:.4f} seconds")

Numba execution time: 0.0486 seconds


## 4. Performance and Scaling Analysis

To perform a comprehensive scaling analysis, we sweep through gradually increasing grid resolutions.
We measure the execution time for all three implementations (Naive, NumPy, and Numba) at each resolution. 

The timing results are stored in a Pandas DataFrame, exported to a CSV file for submission, and finally visualized using a line chart to clearly illustrate the performance differences and scalability of each approach.

In [8]:
import pandas as pd

# Sweep parameter: gradually increasing grid resolutions (N x N)
resolutions = [100, 300, 600, 1000, 1500] 
results = []

print("Starting scaling analysis. This might take a few minutes...")

for N in resolutions:
    print(f"Testing resolution: {N}x{N}")
    
    # 1. Naive
    t0 = time.time()
    _ = mandelbrot_naive(re_min, re_max, im_min, im_max, N, N, max_iter)
    t_naive = time.time() - t0
    
    # 2. NumPy
    t0 = time.time()
    _ = mandelbrot_vectorized(re_min, re_max, im_min, im_max, N, N, max_iter)
    t_numpy = time.time() - t0
    
    # 3. Numba
    t0 = time.time()
    _ = mandelbrot_numba(re_min, re_max, im_min, im_max, N, N, max_iter)
    t_numba = time.time() - t0
    
    # Store results
    results.append({
        'Resolution (NxN)': N,
        'Naive Time (s)': t_naive,
        'NumPy Time (s)': t_numpy,
        'Numba Time (s)': t_numba
    })

# Create DataFrame and export to CSV as required
df_results = pd.DataFrame(results)


display(df_results)

Starting scaling analysis. This might take a few minutes...
Testing resolution: 100x100
Testing resolution: 300x300
Testing resolution: 600x600
Testing resolution: 1000x1000
Testing resolution: 1500x1500


,Resolution (NxN),Naive Time (s),NumPy Time (s),Numba Time (s)
0,100,0.031129,0.006319,0.002207
1,300,0.211326,0.073976,0.017166
2,600,0.840488,0.280531,0.067691
3,1000,2.375049,0.770073,0.184431
4,1500,5.355744,2.120219,0.412972
